# 🎯 Model 1: Career Trajectory & Job Recommendation System
## Alumni-Connect AI/ML Model

### Features:
- Job Placement Prediction (Classification)
- Salary Prediction (Regression)
- Job Recommendation Scoring
- Career Success Prediction

### Algorithms:
1. XGBoost Classifier/Regressor
2. Random Forest (Backup)
3. Neural Network (Advanced)

### Expected Accuracy: 85-94%

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install xgboost lightgbm catboost scikit-learn pandas numpy matplotlib seaborn joblib optuna shap

## 📊 Step 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score
)
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
import lightgbm as lgb
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("✅ Libraries imported successfully!")

## 📁 Step 3: Load Data
Upload your CSV files to Colab or mount Google Drive

In [ ]:
# Option 1: Upload files directly
from google.colab import files
uploaded = files.upload()

# Option 2: Mount Google Drive (Recommended)
# from google.colab import drive
# drive.mount('/content/drive')
# student_df = pd.read_csv('/content/drive/MyDrive/path/to/student_career_data.csv')
# alumni_df = pd.read_csv('/content/drive/MyDrive/path/to/alumni_career_data.csv')

# Load datasets
student_df = pd.read_csv('student_career_data.csv')
alumni_df = pd.read_csv('alumni_career_data.csv')

print(f"✅ Student data shape: {student_df.shape}")
print(f"✅ Alumni data shape: {alumni_df.shape}")

# Display first few rows
print("\n📋 Student Data Sample:")
display(student_df.head())

print("\n📋 Alumni Data Sample:")
display(alumni_df.head())

# Check available features
print("\n📋 Student Data Columns:")
print(student_df.columns.tolist())

## 🔍 Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# -------------------------------
# 🔧 FIX corrupted has_placement column
# -------------------------------

student_df['has_placement'] = pd.to_numeric(
    student_df['has_placement'], 
    errors='coerce'
)

student_df = student_df.dropna(subset=['has_placement'])
student_df['has_placement'] = student_df['has_placement'].astype(int)

# -------------------------------
# Check for missing values
# -------------------------------

print("🔍 Missing Values in Student Data:")
print(student_df.isnull().sum())

# Statistical summary
print("\n📊 Statistical Summary:")
display(student_df.describe())

# -------------------------------
# Class distribution
# -------------------------------

print("\n📈 Placement Distribution:")
print(student_df['has_placement'].value_counts())
print(f"Placement Rate: {student_df['has_placement'].mean() * 100:.2f}%")

# -------------------------------
# Visualizations
# -------------------------------

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# CGPA distribution
axes[0, 0].hist(student_df['cgpa'], bins=20, color='skyblue', edgecolor='black')
axes[0, 0].set_title('CGPA Distribution')
axes[0, 0].set_xlabel('CGPA')

# Skills count distribution
axes[0, 1].hist(student_df['num_skills'], bins=15, color='lightgreen', edgecolor='black')
axes[0, 1].set_title('Number of Skills Distribution')
axes[0, 1].set_xlabel('# Skills')

# Placement by Department
dept_placement = student_df.groupby('department')['has_placement'].mean()
dept_placement.plot(kind='bar', ax=axes[0, 2], color='coral')
axes[0, 2].set_title('Placement Rate by Department')
axes[0, 2].set_ylabel('Placement Rate')

# Package distribution
placed_students = student_df[student_df['has_placement'] == 1]
axes[1, 0].hist(placed_students['placed_package'], bins=15, color='gold', edgecolor='black')
axes[1, 0].set_title('Salary Package Distribution')
axes[1, 0].set_xlabel('Package (INR)')

# CGPA vs Package
axes[1, 1].scatter(placed_students['cgpa'], placed_students['placed_package'], alpha=0.6, color='purple')
axes[1, 1].set_title('CGPA vs Package')
axes[1, 1].set_xlabel('CGPA')
axes[1, 1].set_ylabel('Package (INR)')

# Career Success Score
axes[1, 2].hist(student_df['career_success_score'], bins=20, color='pink', edgecolor='black')
axes[1, 2].set_title('Career Success Score Distribution')
axes[1, 2].set_xlabel('Success Score')

plt.tight_layout()
plt.show()

print("\n✅ EDA Complete!")


## 🔧 Step 5: Feature Engineering

In [ ]:
def engineer_features(df):
    """Create additional features for better predictions"""
    df = df.copy()
    
    # 🔧 Ensure required numeric columns are clean
    numeric_cols = [
        'cgpa', 'num_internships', 'num_certifications',
        'total_internship_months', 'num_skills',
        'avg_skill_proficiency', 'current_year'
    ]
    
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Academic performance category
    df['cgpa_category'] = pd.cut(
        df['cgpa'],
        bins=[0, 6.5, 7.5, 8.5, 10],
        labels=['Low', 'Medium', 'High', 'Excellent']
    )
    
    # Experience score
    df['experience_score'] = (
        df['num_internships'] * 2 +
        df['num_certifications'] +
        df['total_internship_months'] * 0.5
    )
    
    # Profile completeness score
    df['profile_completeness'] = (
        df['has_linkedin'].astype(int) +
        df['has_github'].astype(int) +
        df['has_portfolio'].astype(int)
    ) / 3 * 100
    
    # Skills diversity
    df['skills_diversity'] = (
        df['num_skills'] * df['avg_skill_proficiency'] / 100
    )
    
    # Is final year
    df['is_final_year'] = (df['current_year'] >= 4).astype(int)
    
    # Premium department flag
    premium_depts = ['CSE', 'IT', 'ECE']
    df['is_premium_dept'] = df['department'].isin(premium_depts).astype(int)
    
    return df


# Apply feature engineering
student_df = engineer_features(student_df)

print("✅ Feature engineering complete!")
print(f"New shape: {student_df.shape}")
print(f"\nNew features: {student_df.columns.tolist()[-6:]}")


## 🎯 Step 6: Prepare Data for Training
### Task 1: Placement Prediction (Classification)

In [ ]:
# Select features for placement prediction
feature_columns = [
    'cgpa', 'num_skills', 'num_certifications', 'num_internships',
    'has_linkedin', 'has_github', 'has_portfolio',
    'avg_skill_proficiency', 'total_internship_months',
    'experience_score', 'profile_completeness', 'skills_diversity',
    'is_final_year', 'is_premium_dept'
]

# Encode categorical variables
le_dept = LabelEncoder()
student_df['department_encoded'] = le_dept.fit_transform(student_df['department'])
feature_columns.append('department_encoded')

# -------------------------------
# 🔧 FORCE NUMERIC FOR ALL FEATURES
# -------------------------------
for col in feature_columns:
    student_df[col] = pd.to_numeric(student_df[col], errors='coerce')

# Prepare X and y
X = student_df[feature_columns]
y_placement = student_df['has_placement']

# Handle missing values (now safe)
X = X.fillna(X.median())

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_placement,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y_placement
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Training set: {X_train.shape}")
print(f"✅ Test set: {X_test.shape}")
print(f"✅ Features: {len(feature_columns)}")


## 🚀 Step 7: Train Models - Placement Prediction

In [ ]:
# Initialize models
models = {
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_SEED,
        device='cuda:0'  # XGBoost 3.1+ uses device instead of gpu_id
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=RANDOM_SEED,
        device='gpu'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),
    'Neural Network': MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),
        activation='relu',
        max_iter=500,
        random_state=RANDOM_SEED,
        early_stopping=True
    )
}

# Train and evaluate all models
results = {}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"🔄 Training {name}...")
    
    # Use scaled data for Neural Network, raw for tree-based
    if name == 'Neural Network':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': roc_auc
    }
    
    print(f"✅ {name} Results:")
    print(f"   Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall:    {recall:.4f}")
    print(f"   F1-Score:  {f1:.4f}")
    print(f"   ROC-AUC:   {roc_auc:.4f}")

print("\n" + "="*50)
print("✅ All models trained successfully!")

## 📊 Step 8: Model Comparison & Visualization

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'Precision': [results[m]['precision'] for m in results],
    'Recall': [results[m]['recall'] for m in results],
    'F1-Score': [results[m]['f1_score'] for m in results],
    'ROC-AUC': [results[m]['roc_auc'] for m in results]
})

print("\n📊 Model Comparison:")
display(comparison_df.round(4))

# Find best model
best_model_name = comparison_df.loc[comparison_df['F1-Score'].idxmax(), 'Model']
print(f"\n🏆 Best Model: {best_model_name}")
print(f"   F1-Score: {comparison_df.loc[comparison_df['Model'] == best_model_name, 'F1-Score'].values[0]:.4f}")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot of all metrics
comparison_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']].plot(
    kind='bar', ax=axes[0], width=0.8
)
axes[0].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].legend(loc='lower right')
axes[0].set_ylim([0.7, 1.0])
axes[0].grid(axis='y', alpha=0.3)

# Confusion Matrix for best model
best_model = results[best_model_name]['model']
if best_model_name == 'Neural Network':
    y_pred_best = best_model.predict(X_test_scaled)
else:
    y_pred_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1], cbar=False)
axes[1].set_title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

# Classification report
print(f"\n📋 Classification Report - {best_model_name}:")
print(classification_report(y_test, y_pred_best, target_names=['Not Placed', 'Placed']))

## 🎯 Step 9: Salary Prediction (Regression)

In [ ]:
# Filter only placed students for salary prediction
placed_df = student_df[student_df['has_placement'] == 1].copy()

# Prepare features for salary prediction
X_salary = placed_df[feature_columns]
y_salary = pd.to_numeric(placed_df['placed_package'], errors='coerce')

# 🔧 Force numeric conversion (fix string contamination)
for col in feature_columns:
    X_salary[col] = pd.to_numeric(X_salary[col], errors='coerce')

X_salary = X_salary.fillna(X_salary.median())
y_salary = y_salary.fillna(y_salary.median())

# Split data
X_train_sal, X_test_sal, y_train_sal, y_test_sal = train_test_split(
    X_salary, y_salary,
    test_size=0.2,
    random_state=RANDOM_SEED
)

# Scale features (only if needed)
X_train_sal_scaled = scaler.fit_transform(X_train_sal)
X_test_sal_scaled = scaler.transform(X_test_sal)

# Train regression models
regression_models = {
    'XGBoost': xgb.XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=RANDOM_SEED,
        device='cuda:0'   # ✅ Updated GPU parameter
    ),
    'LightGBM': lgb.LGBMRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=RANDOM_SEED,
        device='gpu'
    ),
    'Random Forest': RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        random_state=RANDOM_SEED,
        n_jobs=-1
    )
}

salary_results = {}

for name, model in regression_models.items():
    print(f"\n🔄 Training {name} for Salary Prediction...")
    
    model.fit(X_train_sal, y_train_sal)
    y_pred_sal = model.predict(X_test_sal)
    
    mse = mean_squared_error(y_test_sal, y_pred_sal)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_sal, y_pred_sal)
    r2 = r2_score(y_test_sal, y_pred_sal)
    
    salary_results[name] = {
        'model': model,
        'rmse': rmse,
        'mae': mae,
        'r2': r2
    }
    
    print(f"✅ {name} Results:")
    print(f"   RMSE: ₹{rmse:,.2f}")
    print(f"   MAE:  ₹{mae:,.2f}")
    print(f"   R²:   {r2:.4f}")

best_sal_model_name = min(salary_results, key=lambda k: salary_results[k]['rmse'])

print(f"\n🏆 Best Salary Prediction Model: {best_sal_model_name}")
print(f"   RMSE: ₹{salary_results[best_sal_model_name]['rmse']:,.2f}")


## 💾 Step 10: Save Models

In [ ]:
# Save best placement model
best_placement_model = results[best_model_name]['model']
joblib.dump(best_placement_model, 'placement_prediction_model.pkl')
joblib.dump(scaler, 'feature_scaler.pkl')
joblib.dump(le_dept, 'department_encoder.pkl')

# Save best salary model
best_salary_model = salary_results[best_sal_model_name]['model']
joblib.dump(best_salary_model, 'salary_prediction_model.pkl')

# Save feature columns
joblib.dump(feature_columns, 'feature_columns.pkl')

print("✅ Models saved successfully!")
print("\n📦 Saved files:")
print("   - placement_prediction_model.pkl")
print("   - salary_prediction_model.pkl")
print("   - feature_scaler.pkl")
print("   - department_encoder.pkl")
print("   - feature_columns.pkl")

# Download files
from google.colab import files
files.download('placement_prediction_model.pkl')
files.download('salary_prediction_model.pkl')
files.download('feature_scaler.pkl')
files.download('department_encoder.pkl')
files.download('feature_columns.pkl')

## 🧪 Step 11: Test Predictions

In [ ]:
# Load models (for testing)
placement_model = joblib.load('placement_prediction_model.pkl')
salary_model = joblib.load('salary_prediction_model.pkl')
scaler = joblib.load('feature_scaler.pkl')

# -------------------------------
# Test with sample student
# -------------------------------

sample_student = {
    'cgpa': 8.5,
    'num_skills': 15,
    'num_certifications': 5,
    'num_internships': 2,
    'has_linkedin': 1,
    'has_github': 1,
    'has_portfolio': 1,
    'avg_skill_proficiency': 85,
    'total_internship_months': 8,
    'department_encoded': 0,  # CSE
    'experience_score': 0,
    'profile_completeness': 0,
    'skills_diversity': 0,
    'is_final_year': 1,
    'is_premium_dept': 1
}

# -------------------------------
# Calculate derived features
# -------------------------------

sample_student['experience_score'] = (
    sample_student['num_internships'] * 2 +
    sample_student['num_certifications'] +
    sample_student['total_internship_months'] * 0.5
)

sample_student['profile_completeness'] = (
    sample_student['has_linkedin'] +
    sample_student['has_github'] +
    sample_student['has_portfolio']
) / 3 * 100

sample_student['skills_diversity'] = (
    sample_student['num_skills'] *
    sample_student['avg_skill_proficiency'] / 100
)

# -------------------------------
# IMPORTANT: Correct feature order
# -------------------------------

feature_columns_order = [
    'cgpa', 'num_skills', 'num_certifications', 'num_internships',
    'has_linkedin', 'has_github', 'has_portfolio',
    'avg_skill_proficiency', 'total_internship_months',
    'experience_score', 'profile_completeness', 'skills_diversity',
    'is_final_year', 'is_premium_dept', 'department_encoded'
]

sample_df = pd.DataFrame([sample_student])[feature_columns_order]

# -------------------------------
# Apply scaling (VERY IMPORTANT)
# -------------------------------

sample_scaled = scaler.transform(sample_df)

# -------------------------------
# Predict placement
# -------------------------------

placement_prob = placement_model.predict_proba(sample_scaled)[0][1]
will_be_placed = placement_model.predict(sample_scaled)[0]

print("\n🎯 Sample Student Prediction:")
print(f"CGPA: {sample_student['cgpa']}")
print(f"Skills: {sample_student['num_skills']}")
print(f"Internships: {sample_student['num_internships']}")

print("\n📊 Placement Prediction:")
print(f"Will be placed: {'✅ YES' if will_be_placed else '❌ NO'}")
print(f"Placement Probability: {placement_prob*100:.2f}%")

# -------------------------------
# Salary Prediction
# -------------------------------

if will_be_placed:
    predicted_salary = salary_model.predict(sample_scaled)[0]
    print(f"\n💰 Expected Salary: ₹{predicted_salary:,.0f}")


## 📈 Step 12: Feature Importance Analysis

In [ ]:
# Get feature importance
if hasattr(best_placement_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': best_placement_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\n📊 Top 10 Most Important Features:")
    display(feature_importance.head(10))
    
    # Visualize
    plt.figure(figsize=(12, 6))
    plt.barh(feature_importance['Feature'].head(15), feature_importance['Importance'].head(15))
    plt.xlabel('Importance')
    plt.title(f'Feature Importance - {best_model_name}', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

print("\n✅ Analysis complete!")

## 🎯 Final Summary

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
import seaborn as sns
import matplotlib.pyplot as plt

best_model = results[best_model_name]['model']

# Use correct test data (scaled if NN)
if best_model_name == 'Neural Network':
    y_pred_best = best_model.predict(X_test_scaled)
    y_proba_best = best_model.predict_proba(X_test_scaled)[:, 1]
else:
    y_pred_best = best_model.predict(X_test)
    y_proba_best = best_model.predict_proba(X_test)[:, 1]

# Compute additional metrics
roc_auc = roc_auc_score(y_test, y_proba_best)
cm = confusion_matrix(y_test, y_pred_best)

print("\n" + "="*60)
print("🎉 MODEL 1: CAREER & JOB RECOMMENDATION - TRAINING COMPLETE!")
print("="*60)

print(f"\n✅ Best Placement Model: {best_model_name}")
print(f"   Accuracy:  {results[best_model_name]['accuracy']*100:.2f}%")
print(f"   Precision: {results[best_model_name]['precision']:.4f}")
print(f"   Recall:    {results[best_model_name]['recall']:.4f}")
print(f"   F1-Score:  {results[best_model_name]['f1_score']:.4f}")
print(f"   ROC-AUC:   {roc_auc:.4f}")

print("\n📊 Confusion Matrix:")
print(cm)

# Confusion matrix heatmap
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred_best))

print(f"\n✅ Best Salary Model: {best_sal_model_name}")
print(f"   RMSE: ₹{salary_results[best_sal_model_name]['rmse']:,.2f}")
print(f"   R²:   {salary_results[best_sal_model_name]['r2']:.4f}")

print(f"\n📦 Total Features: {len(feature_columns)}")
print(f"📊 Training Samples: {len(X_train)}")
print(f"🧪 Test Samples: {len(X_test)}")

print("\n💾 All models and scalers saved successfully!")
print("="*60)
